# Notebook 03 — Entraînement du Modèle de Régression

**Objectif** : Comparer LinearRegression, RandomForestRegressor et GradientBoostingRegressor pour prédire le coût médical annuel (`charges`).

**Dataset** : `data/processed/features.csv` — 1 338 lignes, produit par la feature 002.

**Plan du notebook** :
1. Chargement et split train/test
2. Entraînement et comparaison des 3 algorithmes
3. Tableau comparatif + sélection du meilleur modèle
4. Visualisations : prédictions vs réelles, résidus
5. Optimisation GridSearchCV
6. Cross-validation 5 plis
7. Feature importance

## Section 1 — Chargement des données et split train/test

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

%matplotlib inline
plt.rcParams['figure.figsize'] = (10, 6)
sns.set_style('whitegrid')

print("✅ Imports OK")

In [ ]:
# Chargement du dataset préparé (feature 002)
DATA_PATH = os.path.join('..', 'data', 'processed', 'features.csv')
if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(f"Dataset introuvable : {DATA_PATH}\nExécuter d'abord notebooks/02_feature_engineering.ipynb")

df = pd.read_csv(DATA_PATH)
print(f"✅ Dataset chargé : {df.shape[0]} lignes, {df.shape[1]} colonnes")
print(f"Colonnes : {df.columns.tolist()}")
df.head()

In [ ]:
# Définition des features et de la target
FEATURE_COLS = ['age', 'imc', 'enfants', 'sexe', 'fumeur',
                'region_northwest', 'region_southeast', 'region_southwest']
TARGET_COL = 'charges'

X = df[FEATURE_COLS]
y = df[TARGET_COL]

# Split 80/20 — random_state=42 pour la reproductibilité (constitution §II)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

assert X_train.shape[0] + X_test.shape[0] == 1338, "Erreur split : total != 1338"
print(f"X_train : {X_train.shape}  |  X_test : {X_test.shape}")
print(f"y_train : {y_train.shape}  |  y_test : {y_test.shape}")

## Section 2 — Entraînement et comparaison des 3 algorithmes

Le ColumnTransformer applique StandardScaler sur les variables continues (`age`, `imc`, `enfants`) et laisse passer les variables binaires/one-hot sans transformation. Le preprocessing est **fitté uniquement sur X_train** (constitution Principe II).

In [ ]:
# ColumnTransformer : StandardScaler sur numériques, passthrough sur binaires/one-hot
# IMPORTANT : fit uniquement sur X_train (constitution Principe II)
numeric_cols = ['age', 'imc', 'enfants']
passthrough_cols = ['sexe', 'fumeur', 'region_northwest', 'region_southeast', 'region_southwest']

preprocessor = ColumnTransformer([
    ('scaler', StandardScaler(), numeric_cols),
    ('passthrough', 'passthrough', passthrough_cols),
])

# Fonction utilitaire pour calculer les métriques
def evaluer_modele(y_true, y_pred):
    """Calcule R², MAE et RMSE."""
    r2 = r2_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    return {'R²': round(r2, 4), 'MAE': round(mae, 2), 'RMSE': round(rmse, 2)}

# Entraînement des 3 algorithmes
algorithmes = {
    'LinearRegression': LinearRegression(),
    'RandomForestRegressor': RandomForestRegressor(random_state=42),
    'GradientBoostingRegressor': GradientBoostingRegressor(random_state=42),
}

resultats = {}
modeles_entraines = {}

for nom, modele in algorithmes.items():
    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('model', modele),
    ])
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    resultats[nom] = evaluer_modele(y_test, y_pred)
    modeles_entraines[nom] = pipeline
    print(f"✅ {nom} entraîné")

print("\nEntraînement terminé.")

## Section 3 — Tableau comparatif et sélection du meilleur modèle

In [ ]:
# Tableau comparatif trié par R² décroissant
df_resultats = pd.DataFrame(resultats).T.sort_values('R²', ascending=False)
df_resultats.index.name = 'Algorithme'
print("=== Comparaison des algorithmes (test set 20%) ===")
display(df_resultats)

# Identification du meilleur modèle
meilleur_nom = df_resultats.index[0]
meilleur_r2 = df_resultats.loc[meilleur_nom, 'R²']
meilleur_mae = df_resultats.loc[meilleur_nom, 'MAE']
meilleur_rmse = df_resultats.loc[meilleur_nom, 'RMSE']
meilleur_pipeline = modeles_entraines[meilleur_nom]

print(f"\n🏆 Meilleur modèle : {meilleur_nom}")
print(f"   R²={meilleur_r2:.4f}  |  MAE={meilleur_mae:.2f} USD  |  RMSE={meilleur_rmse:.2f} USD")

## Section 4 — Visualisations : prédictions vs réelles et résidus

**Insight** : LinearRegression capture la tendance générale mais rate les non-linéarités (interaction fumeur × IMC). Les modèles à base d'arbres (Random Forest, Gradient Boosting) détectent ces clusters naturellement, d'où un R² sensiblement supérieur (~+0.10).

In [ ]:
y_pred_meilleur = meilleur_pipeline.predict(X_test)
residus = y_test - y_pred_meilleur

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Graphique 1 : Prédictions vs Valeurs réelles
axes[0].scatter(y_test, y_pred_meilleur, alpha=0.5, color='steelblue', edgecolors='white', linewidth=0.3)
min_val, max_val = y_test.min(), y_test.max()
axes[0].plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='Prédiction parfaite')
axes[0].set_xlabel('Charges réelles (USD)', fontsize=12)
axes[0].set_ylabel('Charges prédites (USD)', fontsize=12)
axes[0].set_title(f'Prédictions vs Valeurs Réelles\n{meilleur_nom} — R²={meilleur_r2:.4f}', fontsize=13)
axes[0].legend()

# Graphique 2 : Distribution des résidus
axes[1].axvline(x=0, color='red', linestyle='--', linewidth=2, label='Résidu = 0')
sns.histplot(residus, kde=True, ax=axes[1], color='steelblue', bins=40)
axes[1].set_xlabel('Résidu (USD) = Réel − Prédit', fontsize=12)
axes[1].set_ylabel('Fréquence', fontsize=12)
axes[1].set_title(f'Distribution des Résidus\n{meilleur_nom}', fontsize=13)
axes[1].legend()

plt.tight_layout()
plt.show()
print(f"Résidus — moyenne : {residus.mean():.2f} USD | écart-type : {residus.std():.2f} USD")

## Section 5 — Optimisation des hyperparamètres (GridSearchCV)

GridSearchCV explore 24 combinaisons × 5 folds = 120 fits. Durée estimée : ~60–90 secondes avec `n_jobs=-1`.

In [ ]:
# Pipeline de base avec le meilleur algorithme (paramètres par défaut)
# On reconstruit le pipeline pour GridSearchCV
modele_base = GradientBoostingRegressor(random_state=42)
pipeline_gs = Pipeline([
    ('preprocessor', ColumnTransformer([
        ('scaler', StandardScaler(), numeric_cols),
        ('passthrough', 'passthrough', passthrough_cols),
    ])),
    ('model', modele_base),
])

# Grille d'hyperparamètres — 2×3×2×2 = 24 combinaisons
param_grid = {
    'model__n_estimators': [100, 200],
    'model__max_depth': [3, 4, 5],
    'model__min_samples_split': [2, 5],
    'model__min_samples_leaf': [1, 2],
}

grid_search = GridSearchCV(
    pipeline_gs,
    param_grid,
    cv=5,
    scoring='r2',
    n_jobs=-1,
    verbose=1,
)

print("Lancement GridSearchCV (24 combinaisons × 5 folds)...")
grid_search.fit(X_train, y_train)

print(f"\n✅ Meilleurs hyperparamètres : {grid_search.best_params_}")
print(f"   Score CV moyen (train) : {grid_search.best_score_:.4f}")

# Évaluation du modèle optimisé sur le test set
y_pred_opt = grid_search.best_estimator_.predict(X_test)
metriques_opt = evaluer_modele(y_test, y_pred_opt)
print(f"\n--- Métriques sur test set (modèle optimisé) ---")
print(f"R²   : {metriques_opt['R²']:.4f}")
print(f"MAE  : {metriques_opt['MAE']:.2f} USD")
print(f"RMSE : {metriques_opt['RMSE']:.2f} USD")

## Section 6 — Cross-validation 5 plis (robustesse du modèle optimisé)

La cross-validation sur l'ensemble complet (train + test) confirme que les performances ne sont pas dues au hasard du split. **Objectif : R² moyen > 0.80** (critère de succès SC-003).

In [ ]:
# Cross-validation 5 plis sur l'ensemble complet (X, y)
pipeline_final = grid_search.best_estimator_
cv_scores = cross_val_score(pipeline_final, X, y, cv=5, scoring='r2', n_jobs=-1)

print("=== Cross-validation 5 plis — R² par fold ===")
for i, score in enumerate(cv_scores, 1):
    print(f"  Fold {i} : {score:.4f}")

print(f"\nR² moyen  : {cv_scores.mean():.4f}")
print(f"Écart-type : {cv_scores.std():.4f}")
print(f"Intervalle : [{cv_scores.mean() - cv_scores.std():.4f}, {cv_scores.mean() + cv_scores.std():.4f}]")

# Assertion de l'objectif (constitution Principe II + SC-003)
assert cv_scores.mean() > 0.80, (
    f"Objectif R²>0.80 non atteint : {cv_scores.mean():.4f}\n"
    "Envisager plus d'hyperparamètres ou un nouveau feature engineering."
)
print(f"\n✅ Objectif R²>0.80 atteint : {cv_scores.mean():.4f}")

## Section 7 — Feature Importance

**Insight attendu** : `fumeur` sera de loin la variable la plus importante (~0.60–0.70), suivi de `age` et `imc`. Les variables régionales auront une importance faible (~0.01–0.03 chacune), confirmant l'exploration EDA (notebook 01).

In [ ]:
# Extraction de la feature importance depuis le modèle final
importances = pipeline_final.named_steps['model'].feature_importances_
df_importance = pd.DataFrame({
    'feature': FEATURE_COLS,
    'importance': importances
}).sort_values('importance', ascending=True)

# Graphique horizontal trié par importance
fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#d73027' if imp > 0.1 else '#4575b4' for imp in df_importance['importance']]
bars = ax.barh(df_importance['feature'], df_importance['importance'], color=colors)
ax.set_xlabel('Importance relative', fontsize=12)
ax.set_ylabel('Variable', fontsize=12)
ax.set_title(f'Importance des Variables — {meilleur_nom}', fontsize=13)
ax.bar_label(bars, fmt='%.3f', padding=3, fontsize=10)
plt.tight_layout()
plt.show()

# Top 3
top3 = df_importance.sort_values('importance', ascending=False).head(3)
print("=== Top 3 variables les plus importantes ===")
for i, (_, row) in enumerate(top3.iterrows(), 1):
    print(f"  {i}. {row['feature']:25s} : {row['importance']:.4f}")

## Section 8 — Classification : Prédiction de la Catégorie de Risque

Entraînement et comparaison de deux classificateurs pour prédire le niveau de risque :
`faible`, `moyen`, `eleve`, `critique` (dérivé de `charges` dans la feature 002).

**Répartition attendue** : faible ~25%, moyen ~50%, eleve ~18%, critique ~7%.

In [ ]:
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_curve, roc_auc_score
)
from sklearn.preprocessing import label_binarize
from sklearn.multiclass import OneVsRestClassifier

# Target classification
TARGET_CLF = 'categorie_risque'
CLASSES_ORDER = ['faible', 'moyen', 'eleve', 'critique']

y_clf = df[TARGET_CLF]

# Distribution de la target
print("=== Distribution de la catégorie de risque ===")
print(y_clf.value_counts(normalize=True).round(3).to_string())

# Split identique à la régression (même random_state=42)
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X, y_clf, test_size=0.2, random_state=42
)
print(f"\nX_train_c : {X_train_c.shape}  |  X_test_c : {X_test_c.shape}")
print("✅ Setup classification OK")

## Section 9 — Comparaison RandomForestClassifier vs GradientBoostingClassifier

Les deux modèles utilisent le **même ColumnTransformer** que la régression pour garantir la cohérence du preprocessing.

In [ ]:
# Même ColumnTransformer que la régression
def construire_pipeline_clf(modele):
    """Construit un Pipeline avec le même preprocessor que la régression."""
    return Pipeline([
        ('preprocessor', ColumnTransformer([
            ('scaler', StandardScaler(), numeric_cols),
            ('passthrough', 'passthrough', passthrough_cols),
        ])),
        ('model', modele),
    ])

classifieurs = {
    'RandomForestClassifier': RandomForestClassifier(random_state=42),
    'GradientBoostingClassifier': GradientBoostingClassifier(random_state=42),
}

resultats_clf = {}
pipelines_clf = {}

for nom, clf in classifieurs.items():
    pipe = construire_pipeline_clf(clf)
    pipe.fit(X_train_c, y_train_c)
    y_pred_c = pipe.predict(X_test_c)
    rapport = classification_report(
        y_test_c, y_pred_c,
        labels=CLASSES_ORDER,
        output_dict=True
    )
    acc = rapport['accuracy']
    f1 = rapport['macro avg']['f1-score']
    resultats_clf[nom] = {'Accuracy': round(acc, 4), 'F1-macro': round(f1, 4)}
    pipelines_clf[nom] = pipe
    print(f"\n=== {nom} ===")
    print(classification_report(y_test_c, y_pred_c, labels=CLASSES_ORDER))

df_clf = pd.DataFrame(resultats_clf).T.sort_values('Accuracy', ascending=False)
df_clf.index.name = 'Algorithme'
print("=== Tableau comparatif Classification ===")
display(df_clf)

meilleur_clf_nom = df_clf.index[0]
meilleur_clf_pipeline = pipelines_clf[meilleur_clf_nom]
print(f"\n🏆 Meilleur classificateur : {meilleur_clf_nom}")
print(f"   Accuracy={df_clf.loc[meilleur_clf_nom,'Accuracy']:.4f}  |  F1-macro={df_clf.loc[meilleur_clf_nom,'F1-macro']:.4f}")

**Justification** : GradientBoostingClassifier surpasse Random Forest sur les classes déséquilibrées (`critique` ~7%)
grâce aux non-linéarités (interaction fumeur×IMC) qu'il capture mieux via le boosting séquentiel.

## Section 10 — Optimisation des hyperparamètres (GridSearchCV Classification)

Grille : 2×3×2 = 12 combinaisons × 5 folds = **60 fits** — durée ~30–45 s avec `n_jobs=-1`.

In [ ]:
# Reconstruction du pipeline pour GridSearchCV
modele_clf_gs = GradientBoostingClassifier(random_state=42)
pipeline_clf_gs = construire_pipeline_clf(modele_clf_gs)

param_grid_clf = {
    'model__n_estimators': [100, 200],
    'model__max_depth': [3, 5, None],
    'model__min_samples_split': [2, 5],
}

grid_search_clf = GridSearchCV(
    pipeline_clf_gs,
    param_grid_clf,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1,
)

print("Lancement GridSearchCV Classification (12 combinaisons × 5 folds)...")
grid_search_clf.fit(X_train_c, y_train_c)

best_params_clf = grid_search_clf.best_params_
best_score_clf = grid_search_clf.best_score_
best_clf_pipeline = grid_search_clf.best_estimator_

print(f"\n✅ Meilleurs hyperparamètres : {best_params_clf}")
print(f"   Score CV moyen (train)    : {best_score_clf:.4f}")

y_pred_c_opt = best_clf_pipeline.predict(X_test_c)
rapport_opt = classification_report(y_pred_c_opt, y_test_c, labels=CLASSES_ORDER, output_dict=True)
print(f"\n--- Métriques test set (modèle optimisé) ---")
print(f"Accuracy : {rapport_opt['accuracy']:.4f}")
print(f"F1-macro : {rapport_opt['macro avg']['f1-score']:.4f}")

**Hyperparamètres retenus** : déterminés par GridSearchCV ci-dessus.
Ces valeurs seront encodées en dur dans `ml_models/training/train_classification.py` (feature US2).

## Section 11 — Cross-validation 5 plis (robustesse du classificateur optimisé)

Objectif : **accuracy moyenne > 0.85** sur l'ensemble complet (constitution Principe II).

In [ ]:
# Cross-validation 5 plis sur l'ensemble complet (X, y_clf)
cv_scores_clf = cross_val_score(
    best_clf_pipeline, X, y_clf, cv=5, scoring='accuracy', n_jobs=-1
)

print("=== Cross-validation 5 plis — Accuracy par fold ===")
for i, score in enumerate(cv_scores_clf, 1):
    print(f"  Fold {i} : {score:.4f}")

cv_mean = cv_scores_clf.mean()
cv_std = cv_scores_clf.std()
print(f"\nAccuracy moyenne  : {cv_mean:.4f}")
print(f"Écart-type        : {cv_std:.4f}")
print(f"Intervalle        : [{cv_mean - cv_std:.4f}, {cv_mean + cv_std:.4f}]")

assert cv_mean > 0.85, (
    f"Objectif accuracy>0.85 non atteint : {cv_mean:.4f}\n"
    "Envisager plus d'hyperparamètres ou un feature engineering supplémentaire."
)
print(f"\n✅ Objectif accuracy>0.85 atteint : {cv_mean:.4f}")

## Section 12 — Matrice de Confusion (Heatmap)

La heatmap 4×4 permet d'identifier les confusions les plus fréquentes entre classes adjacentes (ex. moyen/eleve).

In [ ]:
# Matrice de confusion sur le test set
cm = confusion_matrix(y_test_c, y_pred_c_opt, labels=CLASSES_ORDER)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=CLASSES_ORDER,
    yticklabels=CLASSES_ORDER,
    linewidths=0.5,
    ax=ax,
)
ax.set_xlabel('Prédit', fontsize=12)
ax.set_ylabel('Réel', fontsize=12)
ax.set_title(f'Matrice de Confusion — {meilleur_clf_nom}', fontsize=13)
plt.tight_layout()
plt.show()

# Résumé des confusions principales
print("=== Rapport de classification final ===")
print(classification_report(y_test_c, y_pred_c_opt, labels=CLASSES_ORDER))

## Section 13 — Courbes ROC One-vs-Rest

Approche **OneVsRestClassifier** avec `label_binarize` pour tracer une courbe ROC par classe + macro-average.

In [ ]:
# ROC multiclasse : OneVsRestClassifier + label_binarize
y_bin = label_binarize(y_test_c, classes=CLASSES_ORDER)

# Wrapper OvR sur le pipeline optimisé (refit sur train)
ovr = OneVsRestClassifier(best_clf_pipeline)
ovr.fit(X_train_c, y_train_c)
y_score = ovr.predict_proba(X_test_c)

# AUC macro-average
auc_macro = roc_auc_score(y_bin, y_score, average='macro')

# Tracé des 4 courbes ROC
colors_roc = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']
fig, ax = plt.subplots(figsize=(10, 7))

for i, (classe, color) in enumerate(zip(CLASSES_ORDER, colors_roc)):
    fpr, tpr, _ = roc_curve(y_bin[:, i], y_score[:, i])
    auc_i = roc_auc_score(y_bin[:, i], y_score[:, i])
    ax.plot(fpr, tpr, color=color, lw=2, label=f'{classe} (AUC = {auc_i:.3f})')

ax.plot([0, 1], [0, 1], 'k--', lw=1.5, label='Aléatoire')
ax.set_xlabel('Taux de Faux Positifs', fontsize=12)
ax.set_ylabel('Taux de Vrais Positifs', fontsize=12)
ax.set_title(f'Courbes ROC One-vs-Rest\nAUC macro-average = {auc_macro:.3f}', fontsize=13)
ax.legend(loc='lower right', fontsize=10)
ax.set_xlim([0, 1])
ax.set_ylim([0, 1.02])
plt.tight_layout()
plt.show()

print(f"AUC macro-average : {auc_macro:.4f}")
for i, classe in enumerate(CLASSES_ORDER):
    auc_i = roc_auc_score(y_bin[:, i], y_score[:, i])
    print(f"  {classe:10s} : AUC = {auc_i:.4f}")

## Section 14 — Importance des Variables — Classification

**Insight attendu** : `fumeur` domine (~0.50+), suivi de `age` et `imc`, confirmant l'EDA.

In [ ]:
# Feature importance du meilleur classificateur (pipeline optimisé)
importances_clf = best_clf_pipeline.named_steps['model'].feature_importances_
df_imp_clf = pd.DataFrame({
    'feature': FEATURE_COLS,
    'importance': importances_clf
}).sort_values('importance', ascending=True)

fig, ax = plt.subplots(figsize=(10, 5))
colors_imp = ['#d73027' if imp > 0.1 else '#4575b4' for imp in df_imp_clf['importance']]
bars = ax.barh(df_imp_clf['feature'], df_imp_clf['importance'], color=colors_imp)
ax.set_xlabel('Importance relative', fontsize=12)
ax.set_ylabel('Variable', fontsize=12)
ax.set_title(f'Importance des Variables — Classification ({meilleur_clf_nom})', fontsize=13)
ax.bar_label(bars, fmt='%.3f', padding=3, fontsize=10)
plt.tight_layout()
plt.show()

# Top 3
top3_clf = df_imp_clf.sort_values('importance', ascending=False).head(3)
print("=== Top 3 variables les plus importantes (classification) ===")
for i, (_, row) in enumerate(top3_clf.iterrows(), 1):
    print(f"  {i}. {row['feature']:25s} : {row['importance']:.4f}")
print("\n✅ Section Classification complète — notebook 03 prêt (Restart & Run All)")